## SITCOM- 1953
Karla Peña Ramírez

### Create closure conditions study for TMA

In [ ]:
#Setting packages
import asyncio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
import time
import warnings
import seaborn as sns
import datetime as dt
from scipy import stats
from matplotlib.dates import DateFormatter
from astropy.table import Table, join
from lsst.utils.packages import getEnvironmentPackages
#%matplotlib widget

from astropy.time import Time, TimeDelta
from lsst.daf.butler import Butler 
#from tqdm.notebook import tqdm

# Ignore the many warning messages from ``merge_packed_time_series``
warnings.simplefilter(action="ignore", category=FutureWarning)

#-------Clients
from lsst_efd_client import EfdClient
client = EfdClient("usdf_efd")

#from lsst.summit.utils.efdUtils import makeEfdClient
#client = makeEfdClient()

#from lsst.summit.utils import ConsDbClient
#os.environ["no_proxy"] += ",.consdb"
#url="http://consdb-pq.consdb:8080/consdb"
#consdb=ConsDbClient(url)
#-------Clients

def print_session_info():
    # Time info
    print(f"# Session Info on {time.strftime('%Y-%m-%d at %H:%M:%S %Z', time.localtime(time.time()))}\n")

    # Python info
    print(f"## Python Interpreter\n\nVersion: {sys.version}  \nExecutable: {sys.executable}\n")

    # LSST info
    packages = getEnvironmentPackages(True)
    dev_packages = {"lsst_distrib": packages["lsst_distrib"]}
    dev_packages.update({k: v.split("@")[0] for k, v in packages.items() if "LOCAL" in v})
    print("## Science Pipelines\n\n" + "\n".join(f"{k:<20} {v}" for k, v in dev_packages.items()))

### Historical and 2024-I analysis:
Let's access to specific time ranges and define the sampling rate for the analysis.

In [ ]:
# Historical data retrieval in the time range of interest.
#start = Time("2024-03-01 00:00:00Z", scale="utc")
#end = Time("2025-03-18 00:00:00Z", scale="utc")

#I-2024 window
#start = Time("2024-04-01 00:00:00Z", scale="utc")
#end = Time("2024-06-30 00:00:00Z", scale="utc")

#I-2024 window with Glycol data
start = Time("2024-04-01 00:00:00Z", scale="utc")
end = Time("2024-05-10 00:00:00Z", scale="utc")

#Global sampling
sampling = '1h'

Make the relevant data retrievals:

In [ ]:
#Data retrieval 1: HVAC.dynaleneP05.
query = f"""
SELECT 
    mean(dynTMAsupTS01) AS TMA_Supply_Dynalene_L5, 
    mean(dynTAsupTS03) AS TA_Supply_Dynalene, 
    mean(dynCH01supTS05) AS Chiller1_Supply_Dynalene,
    mean(dynCH02supTS07) AS Chiller2_Supply_Dynalene
FROM "lsst.sal.HVAC.dynaleneP05" 
WHERE time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""

dynaleneP05 = await client.influx_client.query(query)


In [ ]:
#Data retrieval 2: MTMount.cooling
query = f"""
SELECT 
    mean(dynaleneTemperatureAzimuth0001) AS TMA_Azimuth_1, 
    mean(dynaleneTemperatureAzimuth0002) AS TMA_Azimuth_2, 
    mean(dynaleneTemperaturePier0102) AS TMA_Pier_2
FROM "lsst.sal.MTMount.dynaleneCooling" 
WHERE time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""

dynaleneMTMount = await client.influx_client.query(query)


In [ ]:
#Data retrieval 3: MTMount.generalPurposeGlycolWater
query = f"""
SELECT 
    mean(glycolTemperaturePier0001) AS General_Glycol_L6_1,
    mean(glycolTemperaturePier0002) AS General_Glycol_L6_2
FROM "lsst.sal.MTMount.generalPurposeGlycolWater" 
WHERE time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""

GlycolMTMount = await client.influx_client.query(query)


In [ ]:
#Data retrieval 4: MTMount.cooling
query = f"""
SELECT 
    mean(glycolTemperaturePier0101) AS Cold_Glycol_L6
FROM "lsst.sal.MTMount.cooling" 
WHERE time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""

ColdGlycolMTMount = await client.influx_client.query(query)


In [ ]:
##Data retrieval 5: Weather tower relative humidity, temperature and dew point sensors.
#Weather tower humidity sensor.
query = f"""
SELECT 
    mean(relativeHumidityItem) AS Tower_Relative_Humidity
FROM "lsst.sal.ESS.relativeHumidity" 
WHERE salIndex = 301 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_301_humidity = await client.influx_client.query(query)


#Weather tower temperature sensor.
query = f"""
SELECT 
    mean(temperatureItem0) AS Tower_Temperature
FROM "lsst.sal.ESS.temperature" 
WHERE salIndex = 301 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_301_temperature = await client.influx_client.query(query)


#Weather tower dew point sensor.
query = f"""
SELECT 
    mean(dewPointItem) AS Tower_Dew_Point
FROM "lsst.sal.ESS.dewPoint" 
WHERE salIndex = 301 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_301_dewpoint = await client.influx_client.query(query)


In [ ]:
# Data retrieval 6: Environmental. ESS:11[123] temperature and dew points. Also relative humidity for ESS:111
#Camera proxy sensor.
query = f"""
SELECT 
    mean(temperatureItem0) AS ESS_111_temp
FROM "lsst.sal.ESS.temperature" 
WHERE salIndex = 111 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_111_temperature = await client.influx_client.query(query)

query = f"""
SELECT 
    mean(dewPointItem) AS ESS_111_dewp
FROM "lsst.sal.ESS.dewPoint" 
WHERE salIndex = 111 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_111_dewpoint = await client.influx_client.query(query)


query = f"""
SELECT 
    mean(relativeHumidityItem) AS ESS_111_rh
FROM "lsst.sal.ESS.relativeHumidity" 
WHERE salIndex = 111 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_111_humidity = await client.influx_client.query(query)


#M2 proxy sensor.
query = f"""
SELECT 
    mean(temperatureItem0) AS ESS_112_temp
FROM "lsst.sal.ESS.temperature" 
WHERE salIndex = 112 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_112_temperature = await client.influx_client.query(query)


query = f"""
SELECT 
    mean(dewPointItem) AS ESS_112_dewp
FROM "lsst.sal.ESS.dewPoint" 
WHERE salIndex = 112 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_112_dewpoint = await client.influx_client.query(query)


#MTDome proxy sensor.
query = f"""
SELECT 
    mean(temperatureItem0) AS ESS_113_temp
FROM "lsst.sal.ESS.temperature" 
WHERE salIndex = 113 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_113_temperature = await client.influx_client.query(query)


query = f"""
SELECT 
    mean(dewPointItem) AS ESS_113_dewp
FROM "lsst.sal.ESS.dewPoint" 
WHERE salIndex = 113
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_113_dewpoint = await client.influx_client.query(query)


In [ ]:
#Historical
# Data retrieval 7: Environmental. ESS:10[123] temperatures and dew points. Also relative humidity for ESS:101
#Camera proxy sensor.
query = f"""
SELECT 
    mean(temperatureItem0) AS ESS_101_temp
FROM "lsst.sal.ESS.temperature" 
WHERE salIndex = 101 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_101_temperature = await client.influx_client.query(query)

query = f"""
SELECT 
    mean(dewPointItem) AS ESS_101_dewp
FROM "lsst.sal.ESS.dewPoint" 
WHERE salIndex = 101 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_101_dewpoint = await client.influx_client.query(query)

query = f"""
SELECT 
    mean(relativeHumidityItem) AS ESS_101_rh
FROM "lsst.sal.ESS.relativeHumidity" 
WHERE salIndex = 101 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_101_humidity = await client.influx_client.query(query)


#M2 proxy sensor.
query = f"""
SELECT 
    mean(temperatureItem0) AS ESS_102_temp
FROM "lsst.sal.ESS.temperature" 
WHERE salIndex = 102 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_102_temperature = await client.influx_client.query(query)

query = f"""
SELECT 
    mean(dewPointItem) AS ESS_102_dewp
FROM "lsst.sal.ESS.dewPoint" 
WHERE salIndex = 102 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_102_dewpoint = await client.influx_client.query(query)


#MTDome proxy sensor.
query = f"""
SELECT 
    mean(temperatureItem0) AS ESS_103_temp
FROM "lsst.sal.ESS.temperature" 
WHERE salIndex = 103 
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_103_temperature = await client.influx_client.query(query)

query = f"""
SELECT 
    mean(dewPointItem) AS ESS_103_dewp
FROM "lsst.sal.ESS.dewPoint" 
WHERE salIndex = 103
AND time > '{start.isot}Z' 
AND time < '{end.isot}Z' 
GROUP BY time({sampling})
"""
ESS_103_dewpoint = await client.influx_client.query(query)

## Historical consistency on inside sensors  ESS:10[123] and  ESS:11[123]
Let's check the internal consistency among the historical sensors ESS:10[123] and  the current ones ESS:11[123]. This is done with the one year timespan.

In [ ]:
#Inside Dewpoint Consistency
plt.title('Inside Dewpoint Sensor Consistency')

#ax = ESS_111_dewpoint['ESS_111_dewp'].plot(linewidth=1, label='ESS_111')
#ax = ESS_112_dewpoint['ESS_112_dewp'].plot(linewidth=1, label='ESS_112')
#ax = ESS_113_dewpoint['ESS_113_dewp'].plot(linewidth=1, label='ESS_113')
ax = ESS_101_dewpoint['ESS_101_dewp'].plot(linewidth=1, label='ESS_101')
ax = ESS_102_dewpoint['ESS_102_dewp'].plot(linewidth=1, label='ESS_102')
ax = ESS_103_dewpoint['ESS_103_dewp'].plot(linewidth=1, label='ESS_103')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

As can be seen there is a continuity in the trends despite the sensor changes. **The sensor ESS:101(111) is the one with the lower number of gaps shown. ESS:101(111) is chosen as a tracer of the inside dome environment.**

### Inside/Outside consistency for sensors tracing dew points.
Let's know check the dew point information given by the sensors located inside the dome compared to the dew point temperature given by the weather tower sensor.

In [ ]:
#Inside/Outside Dewpoint Consistency
plt.title('Outside/Inside Dew point')
plt.rcParams['figure.figsize'] = [15, 5]

ax = ESS_301_dewpoint['Tower_Dew_Point'].plot(linewidth=1, label='Outside dew point', color='tab:green')
#ax = ESS_111_dewpoint['ESS_111_dewp'].plot(linewidth=1, label='ESS_111', color='tab:blue')
#ax = ESS_112_dewpoint['ESS_112_dewp'].plot(linewidth=1, label='ESS_112', color='tab:blue')
#ax = ESS_113_dewpoint['ESS_113_dewp'].plot(linewidth=1, label='ESS_113', color='tab:blue')
ax = ESS_101_dewpoint['ESS_101_dewp'].plot(linewidth=1, label='Inside dew point', color='tab:orange')
#ax = ESS_102_dewpoint['ESS_102_dewp'].plot(linewidth=1, label='ESS_102', color='tab:orange')
#ax = ESS_103_dewpoint['ESS_103_dewp'].plot(linewidth=1, label='ESS_103', color='tab:orange')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Inside/Outside Temperature Consistency
plt.title('Outside/Inside Temperature')

ax = ESS_301_temperature['Tower_Temperature'].plot(linewidth=1, label='Outside temperature', color='tab:green')
ax = ESS_101_temperature['ESS_101_temp'].plot(linewidth=1, label='Inside temperature', color='tab:orange')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Inside/Outside Relative Humidity Consistency
plt.title('Outside/Inside Relative Humidity')

ax = ESS_301_humidity['Tower_Relative_Humidity'].plot(linewidth=1, label='Outside humidity', color='tab:green')
ax = ESS_101_humidity['ESS_101_rh'].plot(linewidth=1, label='Inside humidity', color='tab:orange')

plt.ylabel('Relative humidity [%]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

All the data is analized only with their mean values in the sampling window. Dew point values coming from the weather station shows lower values compared to the ones from the internal sensor. Despite that, the overall trends of dew point temepratures are alike. The inside dome temperature and relative humidity did not achieve the extreme values the weather tower reports. There was an environmental control. 

Let's see when the ambient temperature crossed the dew point temperature.

In [ ]:
#Ambient temperature crossing the dew point temperature (Inside sensors).
plt.title('T_ambient and T_dewpoint (Inside)')

tempminus10 = ESS_101_temperature['ESS_101_temp']- 10.0
tempminus5 = ESS_101_temperature['ESS_101_temp']- 5.0
ax= ESS_101_dewpoint['ESS_101_dewp'].plot(linewidth=1, label='Inside dew point', color='tab:red')
ax= ESS_101_temperature['ESS_101_temp'].plot(linewidth=1, label='Inside temperature', color='tab:blue', alpha = 0.3)
ax= tempminus5.plot(linewidth=1, label='Inside_temperature - 5°C', color='tab:blue', alpha = 0.5)
ax= tempminus10.plot(linewidth=1, label='Inside_temperature - 10°C', color='tab:blue', alpha = 1.0)

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Ambient temperature crossing the dew point temperature (Outside sensors).
plt.title('T_ambient and T_dewpoint (Outside)')

tempminus10 = ESS_301_temperature['Tower_Temperature']- 10.0
tempminus5 = ESS_301_temperature['Tower_Temperature']- 5.0
ax= ESS_301_dewpoint['Tower_Dew_Point'].plot(linewidth=1, label='Outside dew point', color='tab:red')
ax= ESS_301_temperature['Tower_Temperature'].plot(linewidth=1, label='Outside temperature', color='tab:blue', alpha = 0.3)
ax= tempminus5.plot(linewidth=1, label='Outside temperature - 5°C', color='tab:blue', alpha = 0.5)
ax= tempminus10.plot(linewidth=1, label='Outside temperature - 10°C', color='tab:blue', alpha = 1.0)

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

### Dynalene / Glycol data availability

In [ ]:
#Dynalene/Glycol data availability
plt.title('Supply Dynalene/Glycol')

ax = dynaleneP05['TMA_Supply_Dynalene_L5'].plot(linewidth=1, label='TMA_Supply_Dynalene_L5')
ax = GlycolMTMount['General_Glycol_L6_1'].plot(linewidth=1, label='General_Glycol_L6_1')
ax = GlycolMTMount['General_Glycol_L6_2'].plot(linewidth=1, label='General_Glycol_L6_2')
ax = ColdGlycolMTMount['Cold_Glycol_L6'].plot(linewidth=1, label='Cold_Glycol_L6')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

During 2024-I, only until mid-may we have readings for general and cold glycol. Now let's see the difference between the dew temperature on the dynallene/glycol sensors and the interior dew point using the ESS:101 sensor.

In [ ]:
#Dynalene/Glycol - ESS101
plt.title('Dynalene/Glycol - Inside dew point')

deltaTMA = dynaleneP05['TMA_Supply_Dynalene_L5'] - ESS_101_dewpoint['ESS_101_dewp']
ax= deltaTMA.plot(linewidth=1, label='TMA_Supply_Dynalene_L5 - Inside dew point', color='tab:blue')

deltaGenGly1 = GlycolMTMount['General_Glycol_L6_1'] - ESS_101_dewpoint['ESS_101_dewp']
deltaGenGly2 = GlycolMTMount['General_Glycol_L6_2'] - ESS_101_dewpoint['ESS_101_dewp']
ax= deltaGenGly1.plot(linewidth=1, label='General_Glycol_L6_1 - Inside dew point', color='tab:red')
ax= deltaGenGly2.plot(linewidth=1, label='General_Glycol_L6_2 - Inside dew point', color='tab:orange')

deltaColdGly = ColdGlycolMTMount['Cold_Glycol_L6'] - ESS_101_dewpoint['ESS_101_dewp']
ax= deltaColdGly.plot(linewidth=1, label='Cold_Glycol_L6 - Inside dew point', color='tab:green')

plt.axhline(y=5, color='r', linestyle='--')
plt.ylabel('Delta Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

## ComCam on-sky
Retrieve on-sky time ranges.

In [ ]:
#Identify the repo where all the data lives and the latest consolidated processing collection
#Run > butler query-collections /repo/main "*DRP*w_2025_0*" | grep CHAINED
repo = '/repo/main'
instrument = "LSSTComCam"
collection = 'LSSTComCam/runs/DRP/DP1/w_2025_08/DM-49029'
butler = Butler(repo, collections=collection)
registry = butler.registry

In [ ]:
#Query the metadata for the **`exposure`** dimension, limiting the results to this particular instrument and range of days of observation:
instrument = 'LSSTComCam'
day_obs_start = 20241017
day_obs_end = 20241212
query="instrument='%s' AND day_obs>=%d AND day_obs<=%d" % (instrument, day_obs_start, day_obs_end)
results = registry.queryDimensionRecords('exposure', where=query)

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Stop executing if there are no results returned:
n_results = results.count()

if n_results <= 0:
    raise StopExecution
else:
    print("""There are %d results returned from querying the butler for instrument %s between dates %d and %d (inclusive).""" % 
          (n_results, instrument, day_obs_start, day_obs_end))

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Instantiate a pandas `DataFrame` with useful columns available in the `exposure` dimension:
df_exp = pd.DataFrame(columns=['id', 'obs_id','day_obs', 'seq_num',
                                    'time_start','time_end' ,'type', 'reason', 
                                    'target','filter','zenith_angle',
                                    'expos','ra','dec','skyangle',
                                    'azimuth','zenith','science_program',
                                    'jd','mjd'])

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Read the query results into the new pandas `DataFrame`:
for count, info in enumerate(results):
    
    try:

        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  info.timespan.begin.utc.iso,
                                  info.timespan.end.utc.iso, 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info.timespan.begin.jd, info.timespan.begin.mjd]

    except:
    
        print(">>>   Unexpected error:", sys.exc_info()[0])
        info_timespan_begin_to_string = "2021-01-01 00:00:00.00"
        info_timespan_end_to_string = "2051-01-01 00:00:00.00"
        info_timespan_begin_jd = 0
        info_timespan_begin_mjd = 0
        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  pd.to_datetime(info_timespan_begin_to_string),
                                  pd.to_datetime(info_timespan_end_to_string), 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info_timespan_begin_jd, info_timespan_begin_mjd ]    

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Clean-up the dataframe
#Re-cast the `id`, `day_obs`, and `seq_num` rows as `int`'s:
df_exp = df_exp.astype({"id": int,'day_obs': int,'seq_num':int})
# Replace `NaN`'s in the `ra` and `dec` columns with zero.  
df_exp['ra'] = df_exp['ra'].fillna(0)
df_exp['dec'] = df_exp['dec'].fillna(0)
#Select only on-sky data
df_exp.type.unique()
df_open = df_exp[(df_exp.type == 'science') | (df_exp.type == 'cwfs') | (df_exp.type == 'focus') | (df_exp.type == 'acq')| (df_exp.type == 'flat')]
df_open
# Look at columns for the (exposure/visit) id, start, end
df_open[['obs_id', 'day_obs', 'time_start','time_end']]

In [ ]:
#Group data by dayobs identifying the extremme dates for the exposures.
result_on_sky = df_open.groupby('day_obs').agg({
    'time_start': 'min',
    'time_end': 'max'
}).reset_index()

In [ ]:
def read_time_data():
    df = result_on_sky
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['time_end'] = pd.to_datetime(df['time_end'])
    return df
    
def get_time_range(df, day_obs):
    day_data = df[df['day_obs'] == day_obs].iloc[0]
    return day_data['time_start'], day_data['time_end']

Retrieve telemetry for the entire ComCam on-sky period as well for specific dates:

In [ ]:
#Assited by claude 3.5 sonnet
#Dynalene HVAC
async def get_dynalene_data(client, day_obs, sampling="1h"):
    """Get dynalene data for a specific day_obs."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(dynTMAsupTS01) AS TMA_Supply_Dynalene_L5, 
        mean(dynTMAretTS02) AS TMA_Return_Dynalene, 
        mean(dynCH01supTS05) AS TMA_Chiller_1
    FROM "lsst.sal.HVAC.dynaleneP05" 
    WHERE time > '{start_time.isoformat()}Z' 
    AND time < '{end_time.isoformat()}Z' 
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_dynalene_data(client, sampling="1h"):
    """Get dynalene data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(dynTMAsupTS01) AS TMA_Supply_Dynalene_L5, 
            mean(dynTMAretTS02) AS TMA_Return_Dynalene, 
            mean(dynCH01supTS05) AS TMA_Chiller_1
        FROM "lsst.sal.HVAC.dynaleneP05" 
        WHERE time > '{start_time.isoformat()}Z' 
        AND time < '{end_time.isoformat()}Z' 
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()
      

#Dynalene MTMount
async def get_dynalene_mtmount(client, day_obs, sampling="1h"):
    """Get MTMount dynalene data for a specific day_obs."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(dynaleneTemperatureAzimuth0001) AS TMA_Azimuth_1,
        mean(dynaleneTemperatureAzimuth0002) AS TMA_Azimuth_2,
        mean(dynaleneTemperaturePier0102) AS TMA_Pier_2
    FROM "lsst.sal.MTMount.dynaleneCooling"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_dynalene_mtmount(client, sampling="1h"):
    """Get MTMount dynalene data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(dynaleneTemperatureAzimuth0001) AS TMA_Azimuth_1,
            mean(dynaleneTemperatureAzimuth0002) AS TMA_Azimuth_2,
            mean(dynaleneTemperaturePier0102) AS TMA_Pier_2
        FROM "lsst.sal.MTMount.dynaleneCooling"
        WHERE time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()


#Camera proxy sensor
async def get_ess_111_temp(client, day_obs, sampling="1h"):
    """Get ESS temperature data for Camera."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS ESS_111_temp
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 111
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_ess_111_temp(client, sampling="1h"):
    """Get ESS temperature data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(temperatureItem0) AS ESS_111_temp
        FROM "lsst.sal.ESS.temperature"
        WHERE salIndex = 111
        AND time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()


async def get_ess_111_dewp(client, day_obs, sampling="1h"):
    """Get ESS dew point data for Camera."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(dewPointItem) AS ESS_111_dewp
    FROM "lsst.sal.ESS.dewPoint"
    WHERE salIndex = 111
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_ess_111_dewp(client, sampling="1h"):
    """Get ESS dew point data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(dewPointItem) AS ESS_111_dewp
        FROM "lsst.sal.ESS.dewPoint"
        WHERE salIndex = 111
        AND time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()
    
    
async def get_ess_111_rh(client, day_obs, sampling="1h"):
    """Get ESS relative humidity data for Camera."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(relativeHumidityItem) AS ESS_111_rh
    FROM "lsst.sal.ESS.relativeHumidity"
    WHERE salIndex = 111
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)

    
async def get_all_ess_111_rh(client, sampling="1h"):
    """Get ESS relative humidity data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(relativeHumidityItem) AS ESS_111_rh
        FROM "lsst.sal.ESS.relativeHumidity"
        WHERE salIndex = 111
        AND time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()
    
#Weather tower sensor.
async def get_ess_301_temp(client, day_obs, sampling="1h"):
    """Get ESS temperature data for Outside."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS Tower_Temperature
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_ess_301_temp(client, sampling="1h"):
    """Get ESS temperature data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(temperatureItem0) AS Tower_Temperature
        FROM "lsst.sal.ESS.temperature"
        WHERE salIndex = 301
        AND time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()
    

async def get_ess_301_dewp(client, day_obs, sampling="1h"):
    """Get ESS dew point data for Outside."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(dewPointItem) AS Tower_Dew_Point
    FROM "lsst.sal.ESS.dewPoint"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_ess_301_dewp(client, sampling="1h"):
    """Get ESS dew point data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(dewPointItem) AS Tower_Dew_Point
        FROM "lsst.sal.ESS.dewPoint"
        WHERE salIndex = 301
        AND time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()


async def get_ess_301_rh(client, day_obs, sampling="1h"):
    """Get ESS relative humidity data for Outside."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(relativeHumidityItem) AS Tower_Relative_Humidity
    FROM "lsst.sal.ESS.relativeHumidity"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)
    

async def get_all_ess_301_rh(client, sampling="1h"):
    """Get ESS relative humidity data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(relativeHumidityItem) AS Tower_Relative_Humidity
        FROM "lsst.sal.ESS.relativeHumidity"
        WHERE salIndex = 301
        AND time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()
    
#Glycol
async def get_mtmount_glycol(client, day_obs, sampling="1h"):
    """Get MTMount Glycol temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(glycolTemperaturePier0001) AS General_Glycol_L6_1,
        mean(glycolTemperaturePier0002) AS General_Glycol_L6_2
    FROM "lsst.sal.MTMount.generalPurposeGlycolWater" 
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_mtmount_glycol(client, sampling="1h"):
    """Get MTMount Glycol data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(glycolTemperaturePier0001) AS General_Glycol_L6_1,
            mean(glycolTemperaturePier0002) AS General_Glycol_L6_2
        FROM "lsst.sal.MTMount.generalPurposeGlycolWater" 
        WHERE time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()
    
#Cold glycol 
async def get_mtmount_cold_glycol(client, day_obs, sampling="1h"):
    """Get MTMount Cold Glycol temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(glycolTemperaturePier0101) AS Cold_Glycol_L6
    FROM "lsst.sal.MTMount.cooling" 
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)

async def get_all_mtmount_cold_glycol(client,  sampling="1h"):
    """Get MTMount Cold Glycol data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            mean(glycolTemperaturePier0101) AS Cold_Glycol_L6
        FROM "lsst.sal.MTMount.cooling" 
        WHERE time > '{start_time.isoformat()}Z'
        AND time < '{end_time.isoformat()}Z'
        GROUP BY time({sampling})
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()

Entire period ComCam on-sky:

In [ ]:
# All period data:
sampling_lowres = '1m' 

## Dynalene data
dynalene_all = await get_all_dynalene_data(client, sampling = sampling_lowres)
dynaleneMTMount_all = await get_all_dynalene_mtmount(client, sampling = sampling_lowres)
## ESS data
ESS_111_all_temp = await get_all_ess_111_temp(client, sampling = sampling_lowres)  # Inside temperature
ESS_111_all_dewp = await get_all_ess_111_dewp(client, sampling = sampling_lowres)  # Inside dew point
ESS_111_all_rh = await get_all_ess_111_rh(client, sampling = sampling_lowres)  # Inside relative humidity
ESS_301_all_temp = await get_all_ess_301_temp(client, sampling = sampling_lowres)  # Outside temperature
ESS_301_all_dewp = await get_all_ess_301_dewp(client, sampling = sampling_lowres)  # Outside dew point
ESS_301_all_rh = await get_all_ess_301_rh(client, sampling = sampling_lowres)  # Outside relative humidity
## Glycol data
glycol_all = await get_all_mtmount_glycol(client, sampling = sampling_lowres)  # Glycol temperature
coldglycol_all = await get_all_mtmount_cold_glycol(client, sampling = sampling_lowres)  # Cold glycol temperature


In [ ]:
#Data object creation.
#All days
data_objects_all = [
dynalene_all['TMA_Supply_Dynalene_L5'],
ESS_111_all_temp['ESS_111_temp'],
ESS_111_all_dewp['ESS_111_dewp'],
ESS_111_all_rh['ESS_111_rh'],
ESS_301_all_temp['Tower_Temperature'],
ESS_301_all_dewp['Tower_Dew_Point'],
ESS_301_all_rh['Tower_Relative_Humidity'],  
glycol_all['General_Glycol_L6_1'],
glycol_all['General_Glycol_L6_2'],    
coldglycol_all['Cold_Glycol_L6'],    
]
df_all = pd.concat(data_objects_all, axis=1)

In [ ]:
#Inside/Outside Dewpoint Consistency
plt.title('Outside/Inside Dew point')
plt.rcParams['figure.figsize'] = [15, 5]

ax = df_all['Tower_Dew_Point'].plot(linewidth=1, label='Outside dew point', color='tab:green')
ax = df_all['ESS_111_dewp'].plot(linewidth=1, label='Inside dew point', color='tab:orange')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Inside/Outside Temperature Consistency
plt.title('Outside/Inside Temperature')

ax = df_all['Tower_Temperature'].plot(linewidth=1, label='Outside temperature', color='tab:green')
ax = df_all['ESS_111_temp'].plot(linewidth=1, label='Inside temperature', color='tab:orange')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Inside/Outside Relative Humidity Consistency
plt.title('Outside/Inside Relative Humidity')

ax = df_all['Tower_Relative_Humidity'].plot(linewidth=1, label='Outside humidity', color='tab:green')
ax = df_all['ESS_111_rh'].plot(linewidth=1, label='Inside humidity', color='tab:orange')

plt.ylabel('Relative humidity [%]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Ambient temperature crossing the dew point temperature (Inside sensors)
plt.title('T_ambient and T_dewpoint (Inside)')

tempminus10 = df_all['ESS_111_temp']- 10.0
tempminus5 = df_all['ESS_111_temp']- 5.0
ax= df_all['ESS_111_dewp'].plot(linewidth=1, label='Inside dew point', color='tab:red')
ax= df_all['ESS_111_temp'].plot(linewidth=1, label='Inside temperature', color='tab:blue', alpha = 0.3)
ax= tempminus5.plot(linewidth=1, label='Inside_temperature - 5°C', color='tab:blue', alpha = 0.5)
ax= tempminus10.plot(linewidth=1, label='Inside_temperature - 10°C', color='tab:blue', alpha = 1.0)

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Ambient temperature crossing the dew point temperature (Outside sensors)
plt.title('T_ambient and T_dewpoint (Outside)')

tempminus10 = df_all['Tower_Temperature']- 10.0
tempminus5 = df_all['Tower_Temperature']- 5.0
ax= df_all['Tower_Dew_Point'].plot(linewidth=1, label='Outside dew point', color='tab:red')
ax= df_all['Tower_Temperature'].plot(linewidth=1, label='Outside temperature', color='tab:blue', alpha = 0.3)
ax= tempminus5.plot(linewidth=1, label='Outside temperature - 5°C', color='tab:blue', alpha = 0.5)
ax= tempminus10.plot(linewidth=1, label='Outside temperature - 10°C', color='tab:blue', alpha = 1.0)

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Dynalene/Glycol data availability
plt.title('Supply Dynalene/Glycol')

ax = df_all['TMA_Supply_Dynalene_L5'].plot(linewidth=1, label='TMA_Supply_Dynalene_L5')
ax = df_all['General_Glycol_L6_1'].plot(linewidth=1, label='General_Glycol_L6_1')
ax = df_all['General_Glycol_L6_2'].plot(linewidth=1, label='General_Glycol_L6_2')
ax = df_all['Cold_Glycol_L6'].plot(linewidth=1, label='Cold_Glycol_L6')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Dynalene/Glycol - ESS111
plt.title('Dynalene/Glycol - Inside dew point')

deltaTMA = df_all['TMA_Supply_Dynalene_L5'] - df_all['ESS_111_dewp']
ax= deltaTMA.plot(linewidth=1, label='TMA_Supply_Dynalene_L5 - Inside dew point', color='tab:blue')

deltaGenGly1 = df_all['General_Glycol_L6_1'] - df_all['ESS_111_dewp']
deltaGenGly2 = df_all['General_Glycol_L6_2'] - df_all['ESS_111_dewp']
ax= deltaGenGly1.plot(linewidth=1, label='General_Glycol_L6_1 - Inside dew point', color='tab:red')
ax= deltaGenGly2.plot(linewidth=1, label='General_Glycol_L6_2 - Inside dew point', color='tab:orange')

deltaColdGly = df_all['Cold_Glycol_L6'] - df_all['ESS_111_dewp']
ax= deltaColdGly.plot(linewidth=1, label='Cold_Glycol_L6 - Inside dew point', color='tab:green')

plt.axhline(y=5, color='r', linestyle='--')
plt.ylabel('Delta Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

Specific day during ComCam on-sky:

In [ ]:
# Specific day data:
sampling_lowres = '1m' 
day_to_check = '20241119'

## Dynalene data
dynalene = await get_dynalene_data(client, sampling = sampling_lowres, day_obs=day_to_check)
dynaleneMTMount = await get_dynalene_mtmount(client, sampling = sampling_lowres, day_obs=day_to_check)
## ESS temperature data
ESS_111_temp = await get_ess_111_temp(client, sampling = sampling_lowres, day_obs=day_to_check)  # Inside temperature
ESS_111_dewp = await get_ess_111_dewp(client, sampling = sampling_lowres, day_obs=day_to_check)  # Inside dew point
ESS_111_rh = await get_ess_111_rh(client, sampling = sampling_lowres, day_obs=day_to_check)  # Inside relative humidity
ESS_301_temp = await get_ess_301_temp(client, sampling = sampling_lowres, day_obs=day_to_check)  # Outside temperature
ESS_301_dewp = await get_ess_301_dewp(client, sampling = sampling_lowres, day_obs=day_to_check)  # Outside dew point
ESS_301_rh = await get_ess_301_rh(client, sampling = sampling_lowres, day_obs=day_to_check)  # Outside relative humidity
## Glycol data
glycol = await get_mtmount_glycol(client, sampling = sampling_lowres, day_obs=day_to_check)  # Glycol temperature
coldglycol = await get_mtmount_cold_glycol(client, sampling = sampling_lowres, day_obs=day_to_check)  # Cold glycol temperature


In [ ]:
#Data object creation.
#Specific day
data_objects = [
dynalene['TMA_Supply_Dynalene_L5'],
ESS_111_temp['ESS_111_temp'],
ESS_111_dewp['ESS_111_dewp'],
ESS_111_rh['ESS_111_rh'],
ESS_301_temp['Tower_Temperature'],
ESS_301_dewp['Tower_Dew_Point'],
ESS_301_rh['Tower_Relative_Humidity'],  
glycol['General_Glycol_L6_1'],
glycol['General_Glycol_L6_2'],    
coldglycol['Cold_Glycol_L6'],    
]

df = pd.concat(data_objects, axis=1)

In [ ]:
#Inside/Outside Dewpoint Consistency
plt.title('Outside/Inside Dew point on ' + str(day_to_check))
plt.rcParams['figure.figsize'] = [15, 5]

ax = df['Tower_Dew_Point'].plot(linewidth=1, label='Outside dew point', color='tab:green')
ax = df['ESS_111_dewp'].plot(linewidth=1, label='Inside dew point', color='tab:orange')

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Inside/Outside Temperature Consistency
plt.title('Outside/Inside Temperature on ' + str(day_to_check))

ax = df['Tower_Temperature'].plot(linewidth=1, label='Outside temperature', color='tab:green')
ax = df['ESS_111_temp'].plot(linewidth=1, label='Inside temperature', color='tab:orange')

plt.ylabel('Temperature [∘C]')

plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Inside/Outside Relative Humidity Consistency
plt.title('Outside/Inside Relative Humidity on ' + str(day_to_check))

ax = df['Tower_Relative_Humidity'].plot(linewidth=1, label='Outside humidity', color='tab:green')
ax = df['ESS_111_rh'].plot(linewidth=1, label='Inside humidity', color='tab:orange')

plt.ylabel('Relative humidity [%]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Ambient temperature crossing the dew point temperature (Inside sensors)
plt.title('T_ambient and T_dewpoint (Inside) on ' + str(day_to_check))

tempminus10 = df['ESS_111_temp']- 10.0
tempminus5 = df['ESS_111_temp']- 5.0

ax= df['ESS_111_dewp'].plot(linewidth=1, label='Inside dew point', color='tab:red')
ax= df['ESS_111_temp'].plot(linewidth=1, label='Inside temperature', color='tab:blue', alpha = 0.3)
ax= tempminus5.plot(linewidth=1, label='Inside_temperature - 5°C', color='tab:blue', alpha = 0.5)
ax= tempminus10.plot(linewidth=1, label='Inside_temperature - 10°C', color='tab:blue', alpha = 1.0)

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Ambient temperature crossing the dew point temperature (Outside sensors)
plt.title('T_ambient and T_dewpoint (Outside) on ' + str(day_to_check))

tempminus10 = df['Tower_Temperature']- 10.0
tempminus5 = df['Tower_Temperature']- 5.0

ax= df['Tower_Dew_Point'].plot(linewidth=1, label='Outside dew point', color='tab:red')
ax= df['Tower_Temperature'].plot(linewidth=1, label='Outside temperature', color='tab:blue', alpha = 0.3)
ax= tempminus5.plot(linewidth=1, label='Outside temperature - 5°C', color='tab:blue', alpha = 0.5)
ax= tempminus10.plot(linewidth=1, label='Outside temperature - 10°C', color='tab:blue', alpha = 1.0)

plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Dynalene/Glycol - ESS111
plt.title('Dynalene/Glycol - Inside dew point on ' + str(day_to_check))

deltaTMA = df['TMA_Supply_Dynalene_L5'] - df['ESS_111_dewp']
ax= deltaTMA.plot(linewidth=1, label='TMA_Supply_Dynalene_L5 - Inside dew point', color='tab:blue')

deltaGenGly1 = df['General_Glycol_L6_1'] - df['ESS_111_dewp']
deltaGenGly2 = df['General_Glycol_L6_2'] - df['ESS_111_dewp']
ax= deltaGenGly1.plot(linewidth=1, label='General_Glycol_L6_1 - Inside dew point', color='tab:red')
ax= deltaGenGly2.plot(linewidth=1, label='General_Glycol_L6_2 - Inside dew point', color='tab:orange')

deltaColdGly = df['Cold_Glycol_L6'] - df['ESS_111_dewp']
ax= deltaColdGly.plot(linewidth=1, label='Cold_Glycol_L6 - Inside dew point', color='tab:green')

plt.axhline(y=5, color='r', linestyle='--')
plt.ylabel('Delta Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

### Formula comparison

In [ ]:
#Outside Temperature and Humidity
t_dew = ESS_301_temperature['Tower_Temperature'] - ((100.0 - ESS_301_humidity['Tower_Relative_Humidity'])/5.0)
plt.title('Dew Temperature')
ax = t_dew.plot(linewidth=1, label='Formula Dew Temperature')
ax = ESS_301_dewpoint['Tower_Dew_Point'].plot(linewidth=1, label='Tower Dew Temperature', alpha=0.5)
plt.ylabel('Temperature [∘C]')
plt.legend(bbox_to_anchor=(1.04, 1), loc="center left")
plt.show()

In [ ]:
#Reproducibility
print_session_info()